# Financial Modeling Prep (FMP) Financial Statements Test
Fetching Apple (AAPL) annual and quarterly income statement, balance sheet, and cash flow.

In [3]:
import requests
import pandas as pd
import json

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)

API_KEY = 'yvHpOy3cJQ6SJA9qXgJtlWgOo3nDKEBr'
BASE_URL = 'https://financialmodelingprep.com/stable'
TICKER = 'AAPL'

def fmp_get(endpoint, params=None):
    p = {'apikey': API_KEY, 'symbol': TICKER}
    if params:
        p.update(params)
    r = requests.get(f'{BASE_URL}/{endpoint}', params=p)
    r.raise_for_status()
    return r.json()

print('Setup complete')

Setup complete


## Annual Statements (last 4 fiscal years)
FMP returns a list of dicts, one per period, newest first.

In [4]:
annual_income_raw = fmp_get('income-statement', {'limit': 4, 'period': 'annual'})
print(f'Periods returned: {[r["date"] for r in annual_income_raw]}')
print(f'Keys available: {list(annual_income_raw[0].keys())}')

Periods returned: ['2025-09-27', '2024-09-28', '2023-09-30', '2022-09-24']
Keys available: ['date', 'symbol', 'reportedCurrency', 'cik', 'filingDate', 'acceptedDate', 'fiscalYear', 'period', 'revenue', 'costOfRevenue', 'grossProfit', 'researchAndDevelopmentExpenses', 'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses', 'sellingGeneralAndAdministrativeExpenses', 'otherExpenses', 'operatingExpenses', 'costAndExpenses', 'netInterestIncome', 'interestIncome', 'interestExpense', 'depreciationAndAmortization', 'ebitda', 'ebit', 'nonOperatingIncomeExcludingInterest', 'operatingIncome', 'totalOtherIncomeExpensesNet', 'incomeBeforeTax', 'incomeTaxExpense', 'netIncomeFromContinuingOperations', 'netIncomeFromDiscontinuedOperations', 'otherAdjustmentsToNetIncome', 'netIncome', 'netIncomeDeductions', 'bottomLineNetIncome', 'eps', 'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil']


In [5]:
# Pivot to match yfinance format: rows = line items, columns = period dates
def fmp_to_df(records, exclude_cols=None):
    exclude = exclude_cols or ['symbol', 'reportedCurrency', 'cik', 'fillingDate',
                               'acceptedDate', 'calendarYear', 'period', 'link', 'finalLink']
    df = pd.DataFrame(records).set_index('date').T
    df = df.drop([c for c in exclude if c in df.index], errors='ignore')
    df.columns.name = 'period'
    df.index.name = 'line_item'
    # Convert to numeric
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

annual_income = fmp_to_df(annual_income_raw)
print('=== ANNUAL INCOME STATEMENT ===')
annual_income

=== ANNUAL INCOME STATEMENT ===


period,2025-09-27,2024-09-28,2023-09-30,2022-09-24
line_item,,,,
filingDate,NaN,NaN,NaN,NaN
fiscalYear,"2,025","2,024","2,023","2,022"
revenue,"416,161,000,000","391,035,000,000","383,285,000,000","394,328,000,000"
costOfRevenue,"220,960,000,000","210,352,000,000","214,137,000,000","223,546,000,000"
grossProfit,"195,201,000,000","180,683,000,000","169,148,000,000","170,782,000,000"
researchAndDevelopmentExpenses,"34,550,000,000","31,370,000,000","29,915,000,000","26,251,000,000"
generalAndAdministrativeExpenses,"27,601,000,000","7,458,000,000",0,0
sellingAndMarketingExpenses,0,"18,639,000,000",0,0
sellingGeneralAndAdministrativeExpenses,"27,601,000,000","26,097,000,000","24,932,000,000","25,094,000,000"


In [ ]:
annual_balance_raw = fmp_get('balance-sheet-statement', {'limit': 4, 'period': 'annual'})
annual_balance = fmp_to_df(annual_balance_raw)
print('=== ANNUAL BALANCE SHEET ===')
annual_balance

In [ ]:
annual_cashflow_raw = fmp_get('cash-flow-statement', {'limit': 4, 'period': 'annual'})
annual_cashflow = fmp_to_df(annual_cashflow_raw)
print('=== ANNUAL CASH FLOW ===')
annual_cashflow

## Quarterly Statements (last 4 quarters)

In [ ]:
quarterly_income_raw = fmp_get('income-statement', {'limit': 4, 'period': 'quarter'})
quarterly_income = fmp_to_df(quarterly_income_raw)
print('=== QUARTERLY INCOME STATEMENT ===')
quarterly_income

In [ ]:
quarterly_balance_raw = fmp_get('balance-sheet-statement', {'limit': 4, 'period': 'quarter'})
quarterly_balance = fmp_to_df(quarterly_balance_raw)
print('=== QUARTERLY BALANCE SHEET ===')
quarterly_balance

In [ ]:
quarterly_cashflow_raw = fmp_get('cash-flow-statement', {'limit': 4, 'period': 'quarter'})
quarterly_cashflow = fmp_to_df(quarterly_cashflow_raw)
print('=== QUARTERLY CASH FLOW ===')
quarterly_cashflow

## Raw JSON — what FMP actually returns
Useful to see what the raw structure looks like before we pivot it.

In [ ]:
print('Raw FMP income statement record (first period):')
print(json.dumps(annual_income_raw[0], indent=2))

## Line items available
Useful for deciding what to display in the app.

In [ ]:
print('--- Income Statement line items ---')
for item in annual_income.index: print(' ', item)

print('\n--- Balance Sheet line items ---')
for item in annual_balance.index: print(' ', item)

print('\n--- Cash Flow line items ---')
for item in annual_cashflow.index: print(' ', item)

## Export to CSV

In [ ]:
annual_income.to_csv('AAPL_fmp_annual_income_stmt.csv')
annual_balance.to_csv('AAPL_fmp_annual_balance_sheet.csv')
annual_cashflow.to_csv('AAPL_fmp_annual_cashflow.csv')
quarterly_income.to_csv('AAPL_fmp_quarterly_income_stmt.csv')
quarterly_balance.to_csv('AAPL_fmp_quarterly_balance_sheet.csv')
quarterly_cashflow.to_csv('AAPL_fmp_quarterly_cashflow.csv')

import os
print('CSVs written:')
for f in sorted(os.listdir('.')):
    if f.startswith('AAPL_fmp') and f.endswith('.csv'):
        print(f'  {f} ({os.path.getsize(f):,} bytes)')